# Part 1: Install dependencies

In [ ]:
pip install ag2

In [ ]:
pip install openai --upgrade pip

In [ ]:
pip install pyautogen

# Part 2: Configure the language model

In [ ]:
import autogen

In [ ]:
import os
deepseek_api_key = os.environ["DEEPSEEK_API_KEY"]
llm_config = {
    "config_list": [
        # {
        #     "model": "GLM-4-Plus",
        #     "api_key": "",
        #     "base_url": "https://open.bigmodel.cn/api/paas/v4/",
        #     "price": [0, 0],
        #     "tags": ["glm"]
        # },        
        {
            "model": "deepseek-chat",
            "api_key": deepseek_api_key,
            "base_url": "https://api.deepseek.com",
            "price": [0, 0],
            "tags": ["deepseek"]
        }
    ]
}

deepseek_config = autogen.filter_config(
    config_list=llm_config["config_list"],
    filter_dict={"tags": ["deepseek"]}
)



# Part 3: Define the agent roles

In [ ]:
user_agent= autogen.ConversableAgent(
    name= "user",
    llm_config= False,
    human_input_mode= "ALWAYS"
)

In [ ]:
def should_end_conversation(message: dict)-> bool:
    """
    Use an agent to determine whether the conversation should end.
    """
    termination_agent= autogen.AssistantAgent(
        name= "termination_checker",
        system_message= "You are Termination Checker, an expert in emotion analysis responsible for deciding when a conversation should end. Reply with exactly 'END_CONVERSATION' if either condition below is met; otherwise, reply with exactly 'CONTINUE_CONVERSATION'. "
                        "Condition 1: The user no longer wants to continue the conversation. "
                        "Condition 2: The user asks for music recommendations. ",
        llm_config= {
            "cache_seed": None,
            **deepseek_config[0]
        },
        human_input_mode= "NEVER"
    )
    user_input= message.get("content", "")
    termination_response= termination_agent.generate_reply(
        messages= [{"content": user_input, "role": "user"}]
    )
    return termination_response== "END_CONVERSATION"

In [ ]:



conversation_guide_preview = autogen.ConversableAgent(
    name="conversation_guide",
    system_message="You are Conversational Guide, an expert conversational guide. Guide the user through a conversation, using Socratic questioning where appropriate, to learn about their mood, music preferences, and other relevant information. "
                     "When asked for a summary, summarize the conversation. ",
    human_input_mode="NEVER",
    is_termination_msg=should_end_conversation ,
    llm_config={
        "cache_seed": None,
        **deepseek_config[0]  # Pass a single model configuration.
    },
    
)

In [ ]:
def summarize_conversation(agent, recipient, summary_args):
    """
    Return the complete conversation as context for the next stage.
    """
    # Read the conversation history.
    messages= agent.chat_messages[recipient]
    # Extract the content of each message.
    contents = [msg["content"] for msg in messages]
    # Combine the conversation into the summary context.
    summary = "\n".join(contents)
    return f"Full conversation between the user and the conversational guide:\n{summary}"

In [ ]:
profile_builder= autogen.ConversableAgent(
    name= "profile_builder",
    system_message= "You are Profile Builder, a psychologist responsible for developing a user profile based on the conversation between the user and the conversational guide. "
                    "A user profile is an abstract description of the user's characteristics, behavior, and preferences. "
                    "More specifically, try to include basic information (such as name, age, and gender), interests and preferences (such as music styles and artists), and psychological characteristics (such as lifestyle and values). "
                    "Do not recommend music to the user; your role is to analyze the user profile. ",
    llm_config= {
        "cache_seed": None,
        **deepseek_config[0]
    },
    human_input_mode= "NEVER",
    description= "Develop a user profile" 
)

In [ ]:
profile_critic_preview= autogen.ConversableAgent(
    name= "profile_critic",
    system_message= "You are Profile Critic, a psychologist responsible for critically reviewing the user profile produced by Profile Builder. Check for missing information or omissions. "
                    "Do not recommend music to the user; your role is to analyze the user profile. "
                    "If the profile is rough because the user has not provided useful information, or if the profile needs no further revision, output 'PROFILE_COMPLETE'. "
                    "If you think the profile can be improved, output 'REFINE_PROFILE'. ",
    llm_config= {
        "cache_seed": None,
        **deepseek_config[0]
    },
    human_input_mode= "NEVER",
    description= "Critically review the user profile"
)

In [ ]:
profile_transitions= {
    profile_builder: [profile_critic_preview],
    profile_critic_preview: [profile_builder]
}

In [ ]:
profile_group= autogen.GroupChat(
    agents= [profile_builder, profile_critic_preview],
    speaker_transitions_type= "allowed",
    allowed_or_disallowed_speaker_transitions= profile_transitions,
    messages= [],
    send_introductions= True
)

In [ ]:
profile_group_manager= autogen.GroupChatManager(
    groupchat= profile_group,
    system_message= "You coordinate the user-profiling group, organizing the two psychologists as they analyze the user profile. ",
    llm_config= {
        "cache_seed": None,
        **deepseek_config[0]
    },
    is_termination_msg= lambda msg: "PROFILE_COMPLETE" in msg["content"]   
)

In [ ]:
music_recommender= autogen.ConversableAgent(
    name= "music_recommender",
    system_message= "You are Music Recommender, an expert music recommender. Recommend music based on the user profile, music preferences, and other relevant information. "
                    "Follow these instructions: 1. Consider both the user's music preferences and their current emotional state as reflected in the user profile. "
                    "2. Keep the recommendations focused instead of covering every music style. "
                    "3. Recommend two music styles, with three tracks for each style. "
                    "4. Include at least one and at most three links under each recommended track. Do not recommend a track if no link is available. "
                    "5. You may use the registered function to retrieve links for the recommended tracks. "
                    "6. When you finish recommending music, return 'RECOMMENDATIONS_COMPLETE'. ",
    llm_config= {
        "cache_seed": None,
        **deepseek_config[0]
    },
    human_input_mode= "NEVER",
    description= "Expert music recommender"
)

# Part 4: Retrieve Last.fm track links

In [ ]:
import requests
import json

In [ ]:
# Search Last.fm by track title and optional artist name.
def search_track(track_name, artist_name= None, limit= 5):
    url= "http://ws.audioscrobbler.com/2.0/"
    params= {
        "method": "track.search",
        "track": track_name,
        "api_key": os.environ["LASTFM_API_KEY"], # Last.fm API key
        "format": "json",
        "limit": limit
    }
    if artist_name:
        params["artist"]= artist_name
    response= requests.get(url, params= params)
    return response.json()

In [ ]:
def get_track_link(track_name: str, artist_name: str= None)-> json:
    track_normalizer= autogen.AssistantAgent(
        name= "track_normalizer",
        system_message= "You are Track Normalizer, a music expert with extensive knowledge of artists worldwide. "
                        "For the given artist and track, identify the artist's nationality. If the artist is Chinese, convert both the artist name and track title to Chinese. "
                        "If the artist is not Chinese, convert both the artist name and track title to English. Remember to convert the track title as well. "
                        "Output only a JSON object in the following format (example): {\"track_name\": \"Hey Jude\", \"artist_name\": \"The Beatles\"}",
        llm_config= {
            "cache_seed": None,
            **deepseek_config[0]
        },
        human_input_mode= "NEVER"
    )
    user_input= f"User-provided track title: {track_name}, Artist name: {artist_name}"
    search_result= track_normalizer.generate_reply(
        messages= [{"content": user_input, "role": "user"}]
    )
    # print(search_result)
    # Parse the search_result string into a dictionary
    try:
        search_result= json.loads(search_result)
    except json.JSONDecodeError:
        print("Failed to parse the search result as JSON.")
        return None
    # 
    normalized_track_name= search_result["track_name"]
    artist_name= search_result["artist_name"]
    search_result= search_track(normalized_track_name, artist_name)
    tracks= search_result.get("results", {}).get("trackmatches", {}).get("track", [])
    if not tracks:
        return
    # Return up to three links, or all available links if fewer are found.
    track_links= [track.get("url") for track in tracks[:3]]
    track_result= {
        "music": normalized_track_name,
        "artist": artist_name,
        "links": track_links
    }
    return track_result

In [ ]:
""" # Example usage
track_name= "Hey Jude"  # Track title
artist_name= "The Beatles"     # Artist name (optional)
track_link= get_track_link(track_name, artist_name)
track_link """

# Part 5: Assemble the recommendation workflow

In [ ]:
import os
from typing import Union
from typing import Literal

In [ ]:
user_agent = autogen.UserProxyAgent(
    name="user",
    code_execution_config={
        "use_docker": False,  # Disable Docker for code execution.
        "work_dir": "coding"  # Working directory for code execution.
    }
)

In [ ]:
def should_end_conversation(message: dict)-> bool:
    """
    Use an agent to determine whether the conversation should end.
    """
    termination_agent= autogen.AssistantAgent(
        name= "termination_checker",
        system_message= "You are Termination Checker, an expert in emotion analysis responsible for deciding when a conversation should end. Reply with exactly 'END_CONVERSATION' if either condition below is met; otherwise, reply with exactly 'CONTINUE_CONVERSATION'. "
                        "Condition 1: The user no longer wants to continue the conversation. "
                        "Condition 2: The user asks for music recommendations. ",
        llm_config= {
            "cache_seed": None,
            **deepseek_config[0]
        },
        human_input_mode= "NEVER"
    )
    user_input= message.get("content", "")
    termination_response= termination_agent.generate_reply(
        messages= [{"content": user_input, "role": "user"}]
    )
    return termination_response== "END_CONVERSATION"

In [ ]:
conversation_guide= autogen.ConversableAgent(
    name= "conversation_guide",
    system_message= "You are Conversational Guide, an expert conversational guide. Guide the user through a conversation, using Socratic questioning where appropriate, to learn about their mood, music preferences, and other relevant information. "
                    "Do not recommend music. Your role is to guide the conversation and gather more information that will help with recommendations. ",
    is_termination_msg= should_end_conversation,
    llm_config= {
        "cache_seed": None,
        **deepseek_config[0]
    },
    human_input_mode= "NEVER" # Never ask for human input
)

In [ ]:
profile_critic= autogen.ConversableAgent(
    name= "profile_critic",
    system_message= "You are Profile Critic, a psychologist responsible for critically reviewing the user profile produced by Profile Builder. Check for missing information or omissions. "
                    "Do not recommend music to the user; your role is to analyze the user profile. "
                    "Do not be overly critical; consider how much information the user has provided. "
                    "Note: If the profile is rough because the user has not provided useful information, or if the profile needs no further revision, output 'PROFILE_COMPLETE'. "    
                    "If you think the profile can be improved, identify areas for improvement based on the conversation and end with 'REFINE_PROFILE'. ",
    llm_config= {
        "cache_seed": None,
        **deepseek_config[0]
    },
    human_input_mode= "NEVER",
    description= "Critically review the user profile"
)

In [ ]:
profile_builder= autogen.ConversableAgent(
    name= "profile_builder",
    system_message= "You are Profile Builder, a psychologist responsible for developing a user profile based on the conversation between the user and the conversational guide. "
                    "A user profile is an abstract description of the user's characteristics, behavior, and preferences. "
                    "More specifically, try to include basic information (such as name, age, and gender), interests and preferences (such as music styles and artists), and psychological characteristics (such as lifestyle and values). "
                    "Try to include basic information (such as name, age, and gender), interests and preferences (such as music styles and artists), and psychological characteristics (such as lifestyle and values). "
                    "Even if the user asks you to recommend music, do not do so; your role is to analyze the user profile. Even if the user asks you to recommend music, do not do so; your role is to analyze the user profile. ",
    llm_config= {
        "cache_seed": None,
        **deepseek_config[0]
    },
    human_input_mode= "NEVER",
    description= "Develop a user profile" 
)

In [ ]:
# The user proxy agent is used for interacting with the music_recommender agent
# and executes tool calls.
link_executor= autogen.ConversableAgent(
    name= "link_executor",
    system_message= "You retrieve music links. You can execute the get_track_link function to obtain links for recommended tracks. Read the following instructions before calling it: ",
    llm_config= {
        "cache_seed": None,
        **deepseek_config[0]
    },
    human_input_mode= "NEVER"
)

In [ ]:
music_recommender= autogen.ConversableAgent(
    name= "music_recommender",
    system_message= "You are Music Recommender, an expert music recommender. Recommend music based on the user profile, music preferences, and other relevant information. "
                    "Follow these instructions: 1. Consider both the user's music preferences and their current emotional state as reflected in the user profile. "
                    "2. Recommend two music styles, with three tracks for each style, unless the user specifies otherwise. "
                    "3. Try to avoid recommending tracks the user is already tired of hearing. "
                    "4. Use the registered function to retrieve links for the recommended tracks. ",
    llm_config= {
        "cache_seed": None,
        **deepseek_config[0]
    },
    human_input_mode= "NEVER",
    description= "Expert music recommender"
)

In [ ]:
# Register the tool signature with the assistant agent.
music_recommender.register_for_llm(name= "get_track_link", description="Retrieve links for a given track and artist.")(get_track_link)
# Register the tool function with the user proxy agent.
link_executor.register_for_execution(name= "get_track_link")(get_track_link)

In [ ]:
autogen.register_function(
    f= get_track_link,
    caller= music_recommender,
    executor= link_executor,
    name= "get_track_link",
    description= "Retrieve music links. "
                 "Input: track title and artist name. "
                 "Output: track and artist information with up to three links."
)

In [ ]:

def select_next_speaker(
    last_speaker: autogen.Agent, 
    groupchat: autogen.GroupChat
) -> Union[autogen.Agent, Literal['auto', 'manual', 'random', 'round_robin'], None]:
    
    # 1. Use the model configuration defined outside this function.
    global deepseek_config
    
    # 2. Cache routing agents to avoid recreating them on each turn.
    if not hasattr(groupchat, '_decision_agents'):
        groupchat._decision_agents = [
            autogen.AssistantAgent(
                name=f"routing_agent_{i}",
                system_message="""You are an expert in language analysis responsible for routing the conversation:
                1. When the user requests music recommendations, reply with 'ROUTE_PROFILER'.
                2. When the user reports that a link is unavailable, reply with 'ROUTE_RECOMMENDER'.
                3. In other cases, reply with 'ROUTE_GUIDE'.
                4. When Profile Critic thinks the profile needs refinement, reply with 'ROUTE_PROFILER'.
                5. When Profile Critic considers the profile complete, reply with 'ROUTE_RECOMMENDER'.""",
                llm_config={
                    "cache_seed": None,
                    **deepseek_config[0]  # Use the configured DeepSeek model.
                },
                human_input_mode="NEVER"
            ) for i in range(1)
        ]

    # 3. Identify the role of the last speaker.
    agent_name = {
        user_agent: "User",
        conversation_guide: "Conversational Guide",
        profile_builder: "Profile Builder", 
        profile_critic: "Profile Critic",
        music_recommender: "Music Recommender"
    }.get(last_speaker, "Tool Executor")

    print(f'Selecting the next speaker. Current speaker: {agent_name}')

    # 4. Route user messages and profile-review feedback.
    if last_speaker in [user_agent, profile_critic]:
        last_msg = groupchat.messages[-1]["content"]
        query = f"Speaker: {agent_name}, Content: {last_msg}"
        
        decisions = []
        for agent in groupchat._decision_agents:
            try:
                response = agent.generate_reply(
                    messages=[{"content": query, "role": "user"}]
                )
                decisions.append(response)
            except Exception as e:
                print(f"Routing agent error: {str(e)}")
                decisions.append("ROUTE_GUIDE")  # Fall back to the conversational guide.

        # Count the routing decisions (one routing agent in this prototype).
        from collections import Counter
        decision_counts = Counter(decisions)
        choice = decision_counts.most_common(1)[0][0]
        print(f'Routing decision counts: {decision_counts}')

        return {
            "ROUTE_PROFILER": profile_builder,
            "ROUTE_RECOMMENDER": music_recommender
        }.get(choice, conversation_guide)

    # 5. Apply fixed transitions for the remaining roles.
    transition_map = {
        conversation_guide: user_agent,
        profile_builder: profile_critic,
        music_recommender: link_executor
    }
    return transition_map.get(last_speaker, user_agent)

In [ ]:
recommendation_group= autogen.GroupChat(
    agents= [user_agent, conversation_guide, profile_builder, profile_critic, music_recommender, link_executor],
    speaker_selection_method= select_next_speaker,
    allow_repeat_speaker= False,
    messages=[],
    max_round=30
    
)

In [ ]:
recommendation_manager= autogen.GroupChatManager(
    groupchat= recommendation_group,
    system_message= "You manage the music recommendation group and are an expert in emotion analysis. Coordinate the conversational guide (who talks with the user), the profiling team (who analyze the user profile), the music recommender, and the link retrieval agent to provide music recommendations. ",
    llm_config= {
        "cache_seed": None,
        **deepseek_config[0]
    },
    human_input_mode= "NEVER"
)

In [ ]:
sequential_chats= [
    {
        "recipient": conversation_guide,
        "summary_method": summarize_conversation,
    },
    {
        "recipient": recommendation_manager,
        "message": "Please generate a profile for the user.",
        "summary_method": "reflection_with_llm"
    },
    {
        "recipient": music_recommender,
        "message": "Please recommend music based on the user profile and music preferences.",
        "summary_method": "reflection_with_llm"
    }
]

In [ ]:
# chat_results= user_agent.initiate_chats(sequential_chats)

In [ ]:

chat_results= user_agent.initiate_chat(
    recommendation_manager,
    summary_method= "last_msg",
    max_turns= 100
)
